In [ ]:
from google.colab import drive
import sqlite3
import os

# 1. Google Drive'ı '/content/drive' yoluna bağla
drive.mount('/content/drive')

# 2. Drive içinde projemiz için bir klasör var mı kontrol et, yoksa oluştur
db_folder = "/content/drive/MyDrive/MeetingMindAI"
if not os.path.exists(db_folder):
    os.makedirs(db_folder)

db_path = os.path.join(db_folder, "meeting_mind.db")

# 3. Veritabanı tablolarını ilk kez oluşturacak fonksiyon
def init_db():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Toplantı özetlerini ve transkriptleri tutacak tablo
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS meetings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            transcript TEXT,
            summary TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

    conn.commit()
    conn.close()
    print(f"🎉 Veritabanı başarıyla hazırlandı ve Drive'a kaydedildi: {db_path}")

init_db()

Mounted at /content/drive
🎉 Veritabanı başarıyla hazırlandı ve Drive'a kaydedildi: /content/drive/MyDrive/MeetingMindAI/meeting_mind.db


In [ ]:
import sqlite3
import pandas as pd

db_path = "/content/drive/MyDrive/MeetingMindAI/meeting_mind.db"

# Veritabanına bağlanıp içindeki verileri Pandas DataFrame ile ekrana basıyoruz
try:
    conn = sqlite3.connect(db_path)
    # SQL sorgusu ile tablodaki tüm verileri çekiyoruz
    query = "SELECT * FROM meetings"
    df = pd.read_sql_query(query, conn)
    conn.close()

    if df.empty:
        print("📂 Tablo şu an mevcut ama içi boş (Henüz yeni bir ses dosyası işlemedik).")
    else:
        print("📊 Kayıtlı Toplantı Özetleri:")
        display(df) # Tabloyu Colab içinde güzel bir arayüzle gösterir
except Exception as e:
    print(f"Bir hata oluştu: {e}")

📊 Kayıtlı Toplantı Özetleri:


,id,transcript,summary,created_at
0,1,"Merhaba sevgili dostum, bugün sana alınganlık...",Konuşmanın Konusu: Alınganlık Sorununun Tanıtı...,2026-06-03 18:27:39
1,2,"Merhaba sevgili dostum, bugün sana alınganlık...",Konuşmanın Konusu: Alınganlık Sorununun Tanıtı...,2026-06-03 18:32:22
2,3,"Merhaba sevgili dostum, bugün sana alınganlık...",Konuşmanın Konusu: Alınganlık Sorununun Tanıtı...,2026-06-03 18:36:53
3,4,"Merhaba sevgili dostum, bugün sana alınganlık...",Konuşmanın Konusu: Alınganlık Sorununun Tanıtı...,2026-06-03 18:55:28
4,5,"Merhaba sevgili dostum, bugün sana alınganlık...",Konuşmanın Konusu: Alınganlık Sorununun Tanıtı...,2026-06-03 19:01:04
5,6,Herkese merhaba. Ben Tenmed'i ekibinden Zelha...,Konuşma Özeti:\n\n1. Konuşmanın Konusu: Proje ...,2026-06-03 19:06:58
6,7,Herkese merhaba. Ben Tenmed'i ekibinden Zelha...,Konuşma Özeti:\n\n1. Konuşmanın Konusu: Proje ...,2026-06-03 19:12:43


In [ ]:
!apt install ffmpeg -y

!pip install -U \
  faster-whisper transformers accelerate bitsandbytes sentencepiece \
  sentence-transformers fastapi uvicorn nest_asyncio pyngrok

!pip install --force-reinstall "numpy<2.0" "pandas<3.0"

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 151.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.5/117.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 131.6 MB/s eta 0:00:00
  Attempting uninstall: uvicorn
    Found existing installation: uvicorn 0.47.0
    Uninstalling uvicorn-0.47.0:
      Successfully uninstall

In [ ]:
import torch
import re
import os
import uuid
import numpy as np

from faster_whisper import WhisperModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
    BitsAndBytesConfig
)

from sentence_transformers import SentenceTransformer

from fastapi import FastAPI, UploadFile, File, HTTPException
from starlette.concurrency import run_in_threadpool

import nest_asyncio
import uvicorn
from pyngrok import ngrok

In [ ]:
# L4 GPU olduğu için bfloat16 kullanabiliyoruz, bu hem hız hem bellek kazandırır
device = "cuda"

model_whisper = WhisperModel(
    "small",              # tiny / base / small / medium / large-v3
    device="cuda" if torch.cuda.is_available() else "cpu",
    compute_type="float16" if torch.cuda.is_available() else "int8"
)


# 2. Llama 8B (Turkish) - 4-bit Quantization
model_id = "ytu-ce-cosmos/turkish-llama-8b-instruct-v0.1"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model_llama = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # L4 üzerinde otomatik yerleşim yapar
)

text_gen_pipeline = pipeline("text-generation", model=model_llama, tokenizer=tokenizer)
!pip install -U sentence-transformers

from sentence_transformers import SentenceTransformer
# Türkçe için optimize edilmiş çok dilli bir model
embedding_model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

print("\n--- TEBRİKLER: L4 GPU Üzerinde Tüm Modeller Hazır! ---")


config.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/22.8k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- TEBRİKLER: L4 GPU Üzerinde Tüm Modeller Hazır! ---


In [ ]:
from fastapi.middleware.cors import CORSMiddleware
app = FastAPI()

chunks_db = []
embeddings_db = []
db_path = "/content/drive/MyDrive/MeetingMindAI/meeting_mind.db" # Veritabanı yolumuz


# --- CORS İZİN AYARLARI (GÜVENLİK KAPISINI AÇIYORUZ) ---
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # React (localhost:5173) dahil her yerin erişmesine izin ver
    allow_credentials=True,
    allow_methods=["*"],  # POST, GET, OPTIONS gibi tüm isteklere izin ver
    allow_headers=["*"],  # Tüm veri başlıklarına izin ver
)
# Benzerlik hesaplama fonksiyonu
def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def chunk_text(text, size=200):
    words = text.split()
    return [
        " ".join(words[i:i+size])
        for i in range(0, len(words), size)
    ]

def clean_text(text):
    text = re.sub(
        r'\b(eee|ııı|şey|yani|falan|filan)\b',
        '',
        text,
        flags=re.IGNORECASE
    )
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# =======================================================
# YENİ ENDPOINT: GEÇMİŞ ÖZETLERİ GETİRME
# =======================================================
@app.get("/history")
async def get_history():
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        # En yeni özet en üstte görünecek şekilde sıralayarak çekiyoruz
        cursor.execute("SELECT id, transcript, summary, created_at FROM meetings ORDER BY created_at DESC")
        rows = cursor.fetchall()
        conn.close()

        history = []
        for row in rows:
            history.append({
                "id": row[0],
                "transcript": row[1],
                "summary": row[2],
                "created_at": row[3]
            })
        return history
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Geçmiş yüklenirken hata oluştu: {str(e)}")


# =======================================================
# GÜNCELLENEN ENDPOINT: SES İŞLEME VE VERİTABANINA KAYDETME
# =======================================================
@app.post("/process-audio")
async def process_audio(file: UploadFile = File(...)):

    temp_filename = f"temp_{uuid.uuid4()}_{file.filename}"

    try:
        # =========================
        # DOSYAYI KAYDET
        # =========================
        with open(temp_filename, "wb") as buffer:
            buffer.write(await file.read())

        # =========================
        # 1. TRANSCRIBE
        # =========================
        segments, info = model_whisper.transcribe(
            temp_filename,
            beam_size=5
        )

        raw_text = " ".join(
            [segment.text for segment in segments]
        )

        cleaned_text = clean_text(raw_text)
        chunks = chunk_text(cleaned_text)

        global chunks_db, embeddings_db

        for chunk in chunks:
            emb = embedding_model.encode(chunk)
            chunks_db.append(chunk)
            embeddings_db.append(emb)

        # =========================
        # 2. PROMPT
        # =========================
        messages = [
            {
                "role": "system",
                "content": """
Sen profesyonel bir toplantı özeti çıkarma asistansın.

Kurallar:
- Sadece metindeki bilgileri kullan.
- Bilgi uydurma.
- Tahmin yapma.
- Genel yorum yapma.
- Ek analiz ekleme.
- Tekrar eden cümle kurma.
- Kısa ve net yaz.
- Özet bittikten sonra devam etme.
- Sadece tek bir özet oluştur.
- "assistant" gibi ifadeler yazma.
"""
            },
            {
                "role": "user",
                "content": f"""
Aşağıdaki konuşma metnini özetle.

Şu formatta cevap ver:

1. Konuşmanın Konusu
2. Önemli Noktalar
3. Alınan Kararlar
4. Yapılacak İşler
5. Sonuç

Eğer konuşmada karar veya yapılacak iş yoksa:
"Belirtilmedi" yaz.

METİN:
{cleaned_text}
"""
            }
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # =========================
        # 3. SUMMARY
        # =========================
        final_output = await run_in_threadpool(
            text_gen_pipeline,
            prompt,
            max_new_tokens=512,
            do_sample=False,
            repetition_penalty=1.15,
            return_full_text=False
        )

        generated_text = final_output[0]["generated_text"].strip()

        # Gereksiz devam eden bölümleri kes
        summary = re.split(
            r'assistant|Bu konuşmayı analiz ettikten sonra|Gerekli bilgiyi topladım',
            generated_text
        )[0].strip()

        # ---------------------------------------------------
        # VERİTABANINA OTOMATİK KAYIT ADIMI
        # ---------------------------------------------------
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()
            cursor.execute(
                "INSERT INTO meetings (transcript, summary) VALUES (?, ?)",
                (raw_text, summary)
            )
            conn.commit()
            conn.close()
            print("💾 Başarılı: Özet ve transkript Google Drive veritabanına işlendi!")
        except Exception as db_err:
            print(f"⚠️ Uyarı: Veritabanına kaydedilirken bir sorun oluştu: {db_err}")
        # ---------------------------------------------------

        return {
            "transcript": raw_text,
            "summary": summary,
            "status": "success"
        }

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=str(e)
        )

    finally:
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
# =======================================================
# 4. ENDPOINT: CHAT (SORU-CEVAP) (/chat)
# =======================================================

import re

# Sistem promptunu garantiye almak için buraya ekliyoruz
SYSTEM_PROMPT = """
Sen bir soru-cevap asistanısın.

Kurallar:
- Yalnızca sorunun cevabını ver.
- Tek ve TAM cümle yaz.
- Yarım cümle kurma.
- Nokta ile bitir.
- Açıklama yapma.
- Liste yapma.
- Fazladan konuşma ekleme.
- Eğer cevap metinde yoksa:
"Bu bilgi metinde bulunmuyor." yaz.
"""

# Hata veren fonksiyonu doğrudan bu hücrenin içine tanımlayarak sorunu kökten çözüyoruz
def extract_answer(text):
    text = re.sub(r'\s+', ' ', text).strip()

    # Gereksiz kalıpları temizle
    banned = [
        "Doğru cevap",
        "Örnek cevap",
        "Açıklama",
        "Cevap:",
        "Yanıt:"
    ]

    for b in banned:
        text = text.replace(b, "")

    # İlk tam cümleyi al
    match = re.search(r'(.+?[.!?])', text)

    if match:
        return match.group(1).strip()

    return text.strip()


@app.post("/chat")
async def chat(question: str):
    global chunks_db, embeddings_db

    if not chunks_db:
        return { "answer": "Sistem hafızasında henüz işlenmiş bir toplantı metni bulunmuyor. Lütfen önce bir ses dosyası yükleyin." }

    question = question.lower().strip()
    q_emb = embedding_model.encode(question)

    scores = []
    for i, emb in enumerate(embeddings_db):
        score = cosine_similarity(q_emb, emb)
        scores.append((score, chunks_db[i]))

    top_chunks = sorted(scores, key=lambda x: x[0], reverse=True)[:5]
    context = "\n".join([c for _, c in top_chunks])

    messages = [{ "role": "system", "content": SYSTEM_PROMPT },
                { "role": "user", "content": f"Bağlam:\n{context}\n\nSoru:\n{question}" }]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(text_gen_pipeline.device)

    output = text_gen_pipeline.model.generate(
        **inputs, max_new_tokens=50, do_sample=False, repetition_penalty=1.15,
        eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = output[0][inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    answer = extract_answer(text).rstrip(";,: ")

    if len(answer) < 3:
        answer = "Bu bilgi metinde bulunmuyor."

    return { "answer": answer }

    # =======================================================
    # CHAT GEÇMİŞİNİ VERİTABANINA KAYDETME ADIMI
    # =======================================================
    try:
        import sqlite3
        # 1. Adımda oluşturduğumuz chat_history tablosuna kaydediyoruz
        conn = sqlite3.connect("/content/drive/MyDrive/MeetingMindAI/meeting_mind.db")
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO chat_history (question, answer) VALUES (?, ?)",
            (question, answer)
        )
        conn.commit()
        conn.close()
        print("💾 Chat mesajı veritabanına kaydedildi!")
    except Exception as db_err:
        print(f"⚠️ Chat kaydedilirken hata oluştu: {db_err}")
    # =======================================================

    return {
        "answer": answer
    }


In [ ]:
# =======================================================
# GEÇMİŞ ÖZETLERİ GETİRME ENDPOINT'İ
# =======================================================
@app.get("/history")
async def get_history():
    db_path = "/content/drive/MyDrive/MeetingMindAI/meeting_mind.db"
    try:
        import sqlite3
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        # En yeni özet en üstte görünecek şekilde sıralayarak çekiyoruz
        cursor.execute("SELECT id, transcript, summary, created_at FROM meetings ORDER BY created_at DESC")
        rows = cursor.fetchall()
        conn.close()

        history = []
        for row in rows:
            history.append({
                "id": row[0],
                "transcript": row[1],
                "summary": row[2],
                "created_at": row[3]
            })
        return history
    except Exception as e:
        from fastapi import HTTPException
        raise HTTPException(status_code=500, detail=f"Geçmiş yüklenirken hata oluştu: {str(e)}")

In [ ]:

# 1. TOKENI BURADA TEKRAR TANIT (Çok Önemli)
NGROK_TOKEN = "****" # Kendi tokenını buraya tırnak içine yapıştır
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Asenkron desteği ve Port temizliği
nest_asyncio.apply()
ngrok.kill()
!fuser -k 8000/tcp

# 3. Yeni tünel başlat
try:
    public_url = ngrok.connect(8000)
    print(f"\n🚀 SUNUCU BAŞARIYLA BAĞLANDI!")
    print(f"🔗 BAĞLANTI: {public_url}")
    print(f"🛠️ TEST (SWAGGER): {public_url}/docs")
    print("-" * 50)
except Exception as e:
    print(f"Ngrok hatası: {e}")

# 4. Sunucuyu başlat
if __name__ == "__main__":
    config = uvicorn.Config(
        app=app,
        host="0.0.0.0",
        port=8000,
        loop="asyncio",
        timeout_keep_alive=600,
        limit_concurrency=5,      # 1'den 5'e çıkardık
        timeout_notify=600
    )
    server = uvicorn.Server(config)
    await server.serve()

INFO:     Started server process [1236]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



🚀 SUNUCU BAŞARIYLA BAĞLANDI!
🔗 BAĞLANTI: NgrokTunnel: "https://wool-overbill-baked.ngrok-free.dev" -> "http://localhost:8000"
🛠️ TEST (SWAGGER): NgrokTunnel: "https://wool-overbill-baked.ngrok-free.dev" -> "http://localhost:8000"/docs
--------------------------------------------------


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'repetition_penalty', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. 

⚠️ Uyarı: Veritabanına kaydedilirken bir sorun oluştu: name 'sqlite3' is not defined
INFO:     176.216.217.135:0 - "POST /process-audio HTTP/1.1" 200 OK
INFO:     176.216.217.135:0 - "OPTIONS /chat?question=Esen%20Han%C4%B1m%20ka%C3%A7%20y%C4%B1ld%C4%B1r%20T%C3%BCrkiye%20de%3F HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=50) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


INFO:     176.216.217.135:0 - "POST /chat?question=Esen%20Han%C4%B1m%20ka%C3%A7%20y%C4%B1ld%C4%B1r%20T%C3%BCrkiye%20de%3F HTTP/1.1" 200 OK
